In [ ]:
import xarray as xr
import numpy as np

from datetime import datetime, UTC

"""
This function is designed to excpand upon the metadata of a given dataset wherever this is known.

One of the tasks was to give every variable a standard_name attribute
Since the CF standard_names table is more developed for atmospheric variables, some chosen standard_names 
were invented, based on the conventions given. Wherever that was the case, it is indicated in a comment attribute.

A lot of the required metadata for ACDD compliance was not available at the time of making this dataset. 
Again, I opted for including the attribute but rather designating it as N/A to highlight that it should be included

"""

# We define some helper functions:
def td_to_iso(td):
    # Computes np.timedelta64 to ISO8601 standard
    # cannot handle variable-length units such as months and years
    
    nanoseconds = td.astype("timedelta64[ns]").astype(int)
    
    seconds = nanoseconds/1e9
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    days, hours = divmod(hours, 24)

    return f"P{int(days)}D{int(hours)}H{int(minutes)}M{seconds}S"

def ts_to_iso(ts):
    #converts a np.datetime64 timestamp to the ISO 8601 format
    return str(ts).replace("-","").replace(":","")+"Z"

def maxdelta(ds, coord):
    # computes the maximum delta for a dataset and a specified coordinate
    return np.max(abs(ds[coord][1:] - ds[coord][:-1]).values)


def get_ds(ds_in, data=None):

    """ Function to create a netCDF/CF and ACDD compliant dataset using xarray, such that results can be written to disk. 

    Args:
        ds_in (xr.Dataset/xr.DataArray)

    Returns:
        ds_out (xr.Dataset/xr.DataArray)
        
    """

    ds_out = ds_in.copy(deep=True).drop_vars(("x", "y"))
    # we drop x, y variables since they are redundant

    # we add general metadata that is specific to the example dataset
    ds_out.attrs["title"] = "KaRIn SSH"
    ds_out.attrs["Conventions"] = "CF-1.11, ACDD-1.3"
    ds_out.attrs["summary"] = "This dataset contains data from the Ka-band Radar Interferometer (KaRIn) on sea-surface height, " \
    "radar cross-section, polarization and resolving power information, in addition to some auxiliary information"
    ds_out.attrs["keywords"] = "SEA SURFACE HEIGHT, RADAR BACKSCATTER, INTERFEROMETERS"
    ds_out.attrs["source"] = "Satellite observations using the Ka-band Radar Interferometer (KaRIn)"

    # Add history
    old_history = ds_out.attrs.get("history", "")
    entry = f"{datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")} - Converted dataset to be CF-1.13 compliant"

    if old_history:
        ds_out.attrs["history"] = ds_out.attrs.get("history", "")
    else:
        ds_out.attrs["history"] = old_history + "\n" + entry


    # we fill in information where it is required
    ds_out["ssh_karin_2"].attrs["standard_name"] = "sea_surface_height_above_reference_ellipsoid"
    ds_out["ssh_karin_2"].attrs["coverage_content_type"] = "physicalMeasurement"
    ds_out["ssh_karin_2"].attrs["ancillary_variables"] = "ssh_karin_uncert"
    ds_out["ssh_karin_uncert"].attrs["coverage_content_type"] = "qualityInformation"

    ds_out["mean_sea_surface_cnescls"].attrs["standard_name"] = "sea_surface_height_above_reference_ellipsoid" 
    ds_out["mean_sea_surface_cnescls"].attrs["coverage_content_type"] = "physicalMeasurement"

    ds_out["bathy"].attrs["standard_name"] = "depth"
    ds_out["bathy"].attrs["long_name"] = "bathymetry"
    ds_out["bathy"].attrs["units"] = "m"
    ds_out["bathy"].attrs["positive"] = "up"

    ds_out["eta_detrend"].attrs["standard_name"] = "sea_surface_height_above_reference_ellipsoid"
    ds_out["eta_detrend"].attrs["long_name"] = "detrended sea-surface height anomaly"
    ds_out["eta_detrend"].attrs["units"] = "m"
    ds_out["eta_detrend"].attrs["comment"] = "This is likely the detrended Sea-surface height anomaly. Original file came without this metadata"

    ds_out["polarization_karin"].attrs["standard_name"] = "polarization"
    ds_out["polarization_karin"].attrs["comment"] = "standard_name does not exist"

    ds_out["total_coherence"].attrs["standard_name"] = "interferometric_coherence"
    ds_out["total_coherence"].attrs["coverage_content_type"] = "physicalMeasurement"
    ds_out["total_coherence"].attrs["comment"] = "standard_name does not exist"

    ds_out["miti_power_250m"].attrs["standard_name"] = "center_beam_spatial_resolution"
    ds_out["miti_power_250m"].attrs["comment"] = "standard_name does not exist"
    ds_out["miti_power_250m"].attrs["coverage_content_type"] = "physicalMeasurement"
    ds_out["miti_power_250m"].attrs["ancillary_variables"] = "miti_power_var_250m"
    ds_out["miti_power_var_250m"].attrs["coverage_content_type"] = "qualityInformation"


    #give all the uncert variables coverage_content_type attributes
    ds_out["latitude"].attrs["ancillary_variables"] = "latitude_uncert"
    ds_out["longitude"].attrs["ancillary_variables"] = "longitude_uncert"  
    ds_out["latitude_uncert"].attrs["coverage_content_type"] = "qualityInformation"
    ds_out["longitude_uncert"].attrs["coverage_content_type"] = "qualityInformation"

    # we mark some of the variables as ancillary variables
    ds_out["sig0_karin_2"].attrs["coverage_content_type"] = "physicalMeasurement"
    ds_out["sig0_karin_2"].attrs["ancillary_variables"] = "sig0_karin_uncert"
    ds_out["sig0_karin_uncert"].attrs["coverage_content_type"] = "qualityInformation"
    #need to correct flag_meanings for sig0_karin_2_qual:
    ds_out["sig0_karin_2_qual"].attrs["flag_meanings"] = ds_out["sig0_karin_2_qual"].attrs["flag_meanings"].replace(",","")


    # we compute the latitude and longitude range of the data
    ds_out.attrs["geospatial_lat_min"] = ds_out.coords["latitude"].min().item()
    ds_out.attrs["geospatial_lat_max"] = ds_out.coords["latitude"].max().item()

    ds_out.attrs["geospatial_lon_min"] = ds_out.coords["longitude"].min().item()
    ds_out.attrs["geospatial_lon_max"] = ds_out.coords["longitude"].max().item()

    ds_out.attrs["geospatial_vertical_min"] = np.min(ds_out["bathy"].values)
    ds_out.attrs["geospatial_vertical_max"] = np.max(ds_out["bathy"].values)
    ds_out.attrs["geospatial_vertical_positive"] = "up"

    
    # We compute the geospatial bounds based on the location of the swath corners
    corner_a = f"{ds_out.coords["longitude"].isel(num_lines = 0, num_pixels=0).item()} {ds_out.coords["latitude"].isel(num_lines = 0, num_pixels=0).item()}, "
    corner_b = f"{ds_out.coords["longitude"].isel(num_lines = 0, num_pixels=-1).item()} {ds_out.coords["latitude"].isel(num_lines = 0, num_pixels=-1).item()}, "
    corner_c = f"{ds_out.coords["longitude"].isel(num_lines = -1, num_pixels=0).item()} {ds_out.coords["latitude"].isel(num_lines = -1, num_pixels=0).item()}, "
    corner_d = f"{ds_out.coords["longitude"].isel(num_lines = -1, num_pixels=-1).item()} {ds_out.coords["latitude"].isel(num_lines = 0, num_pixels=0).item()}, "
    ds_out.attrs["geospatial_bounds"] = ("POLYGON(("+corner_a+corner_b+corner_c+corner_d+corner_a[:-2]+"))")
    ds_out.attrs["geospatial_bounds_crs"] = "OGC:CRS84"


    # We compute time-related metadata
    # Start/stop time, duration and resolution needs to follow the ISO-8601 standard
    # Firstly, when does it start and when does it end:
    ds_out.attrs["time_coverage_start"] = ts_to_iso(np.min(ds_in["time"].values))
    ds_out.attrs["time_coverage_end"] = ts_to_iso(np.max(ds_in["time"].values))

    # Secondly, duration
    ds_out.attrs["time_coverage_duration"] = td_to_iso(np.max(ds_in["time"].values)-np.min(ds_in["time"].values))

    # Thirdly, infer the resolution
    ds_out.attrs["time_coverage_resolution"] = td_to_iso(maxdelta(ds_out, "time"))

    # Flag time_tai as auxiliary
    ds_out["time_tai"].attrs["coverage_content_type"] = "auxiliaryInformation"

    # Fourthly, add time at which it was last modified
    ds_out.attrs["date_modified"] = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    
    # We add another variable containing the relative vorticity, although we do not fill it at first, provided data is not None
    if data:
        ds_out["zeta"] = xr.zeros_like(ds_out["eta_detrend"])
        ds_out["zeta"].attrs["standard_name"] = "ocean_relative_vorticity"
        ds_out["zeta"].attrs["long_name"] = "relative vertical vorticity of ocean"
        ds_out["zeta"].attrs["units"] = "s-1" # 
        ds_out["zeta"].attrs["description"] = "Estimated vertical vorticity component from CoastCurr"
        ds_out["zeta"][:]=data
    
    # Finally, we add some ACDD-recommended metadata (uncomment when available)
    ds_out.attrs["processing_level"] = "Unsmoothed SSH measurement data"
    #ds_out.attrs["publisher_name"] = "N/A"
    #ds_out.attrs["publisher_url"] = "N/A"
    #ds_out.attrs["publisher_email"] = "N/A"
    ds_out.attrs["standard_name_vocabulary"] = "CF Standard Name Table v27"
    #ds_out.attrs["acknowledgment/acknowledgement"] = "N/A"
    #ds_out.attrs["comment"] = "N/A"
    #ds_out.attrs["creator_name"] = "N/A"
    #ds_out.attrs["creator_url"] = "N/A"
    #ds_out.attrs["creator_email"] = "N/A"
    #ds_out.attrs["geospatial_bounds_vertical_crs"] = "N/A"
    #ds_out.attrs["id"] = "N/A"
    #ds_out.attrs["institution"] = "N/A"
    #ds_out.attrs["license"] = "N/A"
    #ds_out.attrs["naming_authority"] = "N/A"
    #ds_out.attrs["project"] = "Not applicable"

    return ds_out


In [23]:
ds_ss=xr.open_dataset('../ds_ss.nc')
ds_ss

<xarray.Dataset> Size: 52MB
Dimensions:                                (num_lines: 2600, num_pixels: 173)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 4MB ...
    longitude                              (num_lines, num_pixels) float64 4MB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/20)
    time                                   (num_lines) datetime64[ns] 21kB ...
    time_tai                               (num_lines) datetime64[ns] 21kB ...
    latitude_uncert                        (num_lines, num_pixels) float64 4MB ...
    longitude_uncert                       (num_lines, num_pixels) float64 4MB ...
    polarization_karin                     (num_lines) object 21kB ...
    ssh_karin_2                            (num_lines, num_pixels) float64 4MB ...
    ...                                     ...
    miti_power_var_250m                    (num_lines, num_pixels) float32 2MB ...
    ancillary_surface_classification_flag  (num_lines, num_pixels) float32 2MB ...
    bathy                                  (num_lines, num_pixels) float64 4MB ...
    eta_detrend                            (num_lines, num_pixels) float64 4MB ...
    x                                      (num_pixels) float64 1kB ...
    y                                      (num_lines) float64 21kB ...
Attributes:
    description:  Unsmoothed SSH measurement data and related information for...

In [24]:
ds_ss_formatted = get_ds(ds_ss)
ds_ss_formatted.to_netcdf("../ds_ss_formatted.nc")
ds_ss_formatted

<xarray.Dataset> Size: 52MB
Dimensions:                                (num_lines: 2600, num_pixels: 173)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 4MB ...
    longitude                              (num_lines, num_pixels) float64 4MB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/18)
    time                                   (num_lines) datetime64[ns] 21kB ...
    time_tai                               (num_lines) datetime64[ns] 21kB ...
    latitude_uncert                        (num_lines, num_pixels) float64 4MB ...
    longitude_uncert                       (num_lines, num_pixels) float64 4MB ...
    polarization_karin                     (num_lines) object 21kB ...
    ssh_karin_2                            (num_lines, num_pixels) float64 4MB ...
    ...                                     ...
    mean_sea_surface_cnescls               (num_lines, num_pixels) float64 4MB ...
    miti_power_250m                        (num_lines, num_pixels) float32 2MB ...
    miti_power_var_250m                    (num_lines, num_pixels) float32 2MB ...
    ancillary_surface_classification_flag  (num_lines, num_pixels) float32 2MB ...
    bathy                                  (num_lines, num_pixels) float64 4MB ...
    eta_detrend                            (num_lines, num_pixels) float64 4MB ...
Attributes: (12/23)
    description:                   Unsmoothed SSH measurement data and relate...
    title:                         KaRIn SSH
    Conventions:                   CF-1.11, ACDD-1.3
    summary:                       This dataset contains data from the Ka-ban...
    keywords:                      SEA SURFACE HEIGHT, RADAR BACKSCATTER, INT...
    source:                        Satellite observations using the Ka-band R...
    ...                            ...
    time_coverage_end:             20231017T122903.820543232Z
    time_coverage_duration:        P0D0H1M36.53201369600001S
    time_coverage_resolution:      P0D0H0M0.03714368S
    date_modified:                 20260702T091730Z
    processing_level:              Unsmoothed SSH measurement data
    standard_name_vocabulary:      CF Standard Name Table v27